In [1]:
import pandas as pd
import numpy as np

In [5]:
df=pd.read_csv('../data/interim/listings_parsed_v7.csv',dtype={'jnc': 'str'})

In [6]:
len(df)

183490

In [7]:
df['group_coarse'].nunique()

22843

In [8]:
from sklearn.model_selection import GroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

BASE = ["area_m2", "age_years", "floor_min", "stories_above", "walk_min_nearest"]
MADORI = ["n_rooms", "has_L", "has_D", "has_K", "has_S", "is_1R"]
FEATURES = BASE + MADORI
CAT = ["municipality"]

splits = list(
    GroupKFold(n_splits=5).split(
        df,
        df["Price_yen"],
        groups=df["group_coarse"]
    )
)

def make_pipe(num_cols=FEATURES, **xgb_params):
    defaults = dict(
        n_estimators=400,
        learning_rate=0.1,
        max_depth=6,
        tree_method="hist",
        device="cuda",
        random_state=0,
        n_jobs=-1,
    )
    defaults.update(xgb_params)

    return Pipeline([
        (
            "preprocessor",
            ColumnTransformer([
                ("num", "passthrough", num_cols),
                ("cat", OneHotEncoder(handle_unknown="ignore"), CAT),
            ])
        ),
        ("regressor", XGBRegressor(**defaults)),
    ])

def fold_maes(pipe_factory):
    X = df[FEATURES + CAT]
    y = df["Price_yen"]

    maes = []
    for tr, te in splits:
        pipe = pipe_factory()
        pipe.fit(X.iloc[tr], y.iloc[tr])
        predictions = pipe.predict(X.iloc[te])
        maes.append(mean_absolute_error(y.iloc[te], predictions))

    return np.asarray(maes)

print(fold_maes(make_pipe).mean())

c:\Users\priya\Desktop\data science try\tokyo-rental-intelligence\.venv\Lib\site-packages\xgboost\core.py:569: UserWarning: [14:26:29] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


11179.41484375


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform
import time

X = df[FEATURES + CAT]
y = df["Price_yen"]

search = RandomizedSearchCV(
    estimator=make_pipe(),
    param_distributions={
        "regressor__max_depth": randint(4, 11),
        "regressor__learning_rate": uniform(0.02, 0.13),
        "regressor__n_estimators": randint(300, 1001),
        "regressor__min_child_weight": randint(1, 11),
        "regressor__subsample": uniform(0.6, 0.4),
        "regressor__colsample_bytree": uniform(0.6, 0.4),
        "regressor__reg_lambda": uniform(0.5, 4.5),
    },
    n_iter=25,
    cv=splits,
    scoring="neg_mean_absolute_error",
    random_state=0,
    refit=False,
    n_jobs=1,
    verbose=2,
)

start = time.time()
search.fit(X, y)
print(f"Runtime: {(time.time() - start) / 60:.2f} minutes")

results = pd.DataFrame({
    "params": search.cv_results_["params"],
    "mean_test_score": -search.cv_results_["mean_test_score"],
    "std_test_score": search.cv_results_["std_test_score"],
    "rank_test_score": search.cv_results_["rank_test_score"],
}).rename(columns={"mean_test_score": "mae"})

print(results.sort_values("mae").head(5))

results.to_csv("../data/interim/xgb_search_results.csv", index=False)


    

Fitting 5 folds for each of 25 candidates, totalling 125 fits
[CV] END regressor__colsample_bytree=0.8195254015709299, regressor__learning_rate=0.11297461762841454, regressor__max_depth=7, regressor__min_child_weight=4, regressor__n_estimators=659, regressor__reg_lambda=2.406446597025071, regressor__subsample=0.8583576452266624; total time=   2.3s
[CV] END regressor__colsample_bytree=0.8195254015709299, regressor__learning_rate=0.11297461762841454, regressor__max_depth=7, regressor__min_child_weight=4, regressor__n_estimators=659, regressor__reg_lambda=2.406446597025071, regressor__subsample=0.8583576452266624; total time=   1.8s
[CV] END regressor__colsample_bytree=0.8195254015709299, regressor__learning_rate=0.11297461762841454, regressor__max_depth=7, regressor__min_child_weight=4, regressor__n_estimators=659, regressor__reg_lambda=2.406446597025071, regressor__subsample=0.8583576452266624; total time=   1.9s
[CV] END regressor__colsample_bytree=0.8195254015709299, regressor__learni

In [11]:
best = results.sort_values("mae").iloc[0]["params"]
params = {k.replace("regressor__", ""): v for k, v in best.items()}
print(params)
tuned = fold_maes(lambda: make_pipe(**params))
diff = tuned - fold_maes(make_pipe)
print(tuned.mean(), diff.mean(), diff.std(), (diff < 0).sum())

{'colsample_bytree': np.float64(0.6977702368006411), 'learning_rate': np.float64(0.04066604587391756), 'max_depth': 9, 'min_child_weight': 6, 'n_estimators': 860, 'reg_lambda': np.float64(1.121823281068762), 'subsample': np.float64(0.6786329446720214)}
10967.05625 -212.35859375 96.54564466091013 5
